# Energy-band diagrams 

Reconstructs the conduction-band edge, electron quasi-Fermi level and valence-band edge from the saved CA-PINN solutions of the pre-bias voltage sweep (`08_Vi_Sweep.ipynb`):

$$E_c(x) = -q\psi(x) - \chi, \qquad E_{Fn}(x) = E_c(x) + k_BT\,\ln\!\left(\frac{n(x)}{N_c}\right), \qquad E_v(x) = E_c(x) - E_g$$

Run this notebook from the repository root.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.constants as sc

In [ ]:
# Material parameters
T = 300.0     # Temperature (K)
Nc = 2.1e24   # Effective density of states in the conduction band (m^-3)
Eg = 1.58     # Band gap (eV)
chi = 4.0     # Electron affinity (eV)

# Pre-bias voltages shown in Fig. 8 and their saved CA-PINN solutions
VI_SWEEP_DIR = os.path.join("results", "Vi_sweep")
CASES = {
    -0.10: "Vi_m0p100V_P0_1e21_phi0.10_tanh_4x40_seed2024",
     0.00: "Vi_p0p000V_P0_1e21_phi0.10_tanh_4x40_seed2024",
     0.25: "Vi_p0p250V_P0_1e21_phi0.10_tanh_4x40_seed2024",
}

OUT_DIR = os.path.join(VI_SWEEP_DIR, "band_diagrams")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def band_edges(case_dir):
    """Return x (nm), Ec, EFn and Ev (eV) from a saved PINN solution.

    psi is stored at x = 0..500 nm and n at the interior points x = 1..499 nm,
    so the two profiles are matched on position before combining them.
    """
    df_psi = pd.read_csv(os.path.join(case_dir, "pinn_psi.csv"))
    df_n = pd.read_csv(os.path.join(case_dir, "pinn_n.csv"))
    df = df_psi.merge(df_n, on="x_nm", how="inner")

    x = df["x_nm"].to_numpy()
    psi = df["psi_V"].to_numpy()
    n = df["n_m3"].to_numpy()

    Ec = -psi - chi
    Ev = Ec - Eg
    Efn = Ec + sc.k * T / sc.e * np.log(n / Nc)
    return x, Ec, Efn, Ev

In [ ]:
def plot_band_diagram(x, Ec, Efn, Ev, out_path):
    fig, ax = plt.subplots(figsize=(3.0, 4.5))

    ax.plot(x, Ec,  lw=1.8, label=r'$E_\mathrm{c}$')
    ax.plot(x, Efn, lw=1.8, label=r'$E_\mathrm{Fn}$')
    ax.plot(x, Ev,  lw=1.8, label=r'$E_\mathrm{v}$')

    ax.set_xlabel('Position (nm)', fontsize=12)
    ax.set_ylabel('Electron energy (eV)', fontsize=12)

    ax.tick_params(axis='both', which='major', direction='in',
                   length=5, width=1.0, labelsize=10)
    ax.tick_params(axis='both', which='minor', direction='in',
                   length=3, width=0.8)
    ax.minorticks_on()

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend at the top, leaving the middle free for the schematic annotations
    ax.legend(loc='upper center', bbox_to_anchor=(0.25, 0.9),
              frameon=False, fontsize=10, ncol=1, handlelength=2.0)

    fig.tight_layout()
    fig.savefig(out_path, dpi=600, bbox_inches='tight')
    plt.show()

In [ ]:
for Vi, tag in CASES.items():
    x, Ec, Efn, Ev = band_edges(os.path.join(VI_SWEEP_DIR, tag))
    name = f"energy_{'m' if Vi < 0 else 'p'}{abs(Vi):.2f}V".replace('.', 'p')
    out_path = os.path.join(OUT_DIR, name + ".pdf")
    print(f"Vi = {Vi:+.2f} V -> {out_path}")
    plot_band_diagram(x, Ec, Efn, Ev, out_path)